In [1]:
!pip install torch transformers datasets tqdm matplotlib

In [2]:
import torch

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using:", device)

Using: mps


In [3]:
import torch
import torch.nn as nn
import time
import re
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
dataset = load_dataset("gsm8k", "main")

train_data = dataset["train"]
test_data = dataset["test"]

In [5]:
teacher_name = "microsoft/phi-2"

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_name)

teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_name,
    torch_dtype=torch.float16
).to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [6]:
def teacher_reason(question):

    prompt = f"""
You are a math expert.

Solve the problem step-by-step.

At the end, write:

Final Answer: <number>

Question:
{question}
"""

    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(device)

    outputs = teacher_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3
    )

    return teacher_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
distilled = []

for sample in tqdm(train_data[:100]):
    reasoning = teacher_reason(sample)

    distilled.append({
        "question": sample,
        "answer": "",   # or generate answer separately
        "reasoning": reasoning
    })

  0%|                                                                                             | 0/2 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
100%|█████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.03s/it]


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [9]:
class StudentModel(nn.Module):

    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=8,
            batch_first=True   # ✅ FIX HERE
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=4
        )

        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        x = self.transformer(x)

        return self.fc(x)

    def generate(self, tokenizer, prompt, max_len=50):

        self.eval()

        tokens = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

        generated = tokens

        for _ in range(max_len):

            out = self.forward(generated)

            next_token = torch.argmax(out[:, -1, :], dim=-1).unsqueeze(1)

            generated = torch.cat([generated, next_token], dim=1)

        return tokenizer.decode(generated[0], skip_special_tokens=True)

In [10]:
vocab_size = tokenizer.vocab_size

student_model = StudentModel(vocab_size).to(device)

optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [11]:
for epoch in range(2):

    for sample in distilled:

        text = sample["question"] + "\n" + sample["reasoning"]

        tokens = tokenizer(text, return_tensors="pt").input_ids.to(device)

        src = tokens[:, :-1]
        tgt = tokens[:, 1:]

        output = student_model(src)

        loss = loss_fn(
            output.reshape(-1, vocab_size),
            tgt.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [12]:
import re

def extract_number(text):

    # First try to extract "Final Answer"
    match = re.search(r"Final Answer:\s*(-?\d+\.?\d*)", text)

    if match:
        return match.group(1)

    # fallback to last number
    nums = re.findall(r"-?\d+\.?\d*", text)

    return nums[-1] if nums else None

In [13]:
teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

In [14]:
import collections

class AgenticStudent:

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def planner(self, question):

        prompt = f"""
Choose strategy.

Question:
{question}

Strategy:
"""

        return self.model.generate(self.tokenizer, prompt)

    def reasoner(self, question, strategy):

        prompt = f"""
Solve step by step.

Strategy:
{strategy}

Question:
{question}

Reasoning:
"""

        return self.model.generate(self.tokenizer, prompt)

    def verifier(self, reasoning):

        prompt = f"""
Give final numeric answer.

Reasoning:
{reasoning}

Answer:
"""

        return self.model.generate(self.tokenizer, prompt)

    def run(self, question):

        strategy = self.planner(question)

        answers = []

        for _ in range(3):

            reasoning = self.reasoner(question, strategy)

            answer = self.verifier(reasoning)

            answers.append(extract_number(answer))

        final = collections.Counter(answers).most_common(1)[0][0]

        return final

In [15]:
correct = 0
total = 50

for i in range(total):

    sample = test_data[i]

    out = teacher_reason(sample["question"])

    pred = extract_number(out)
    gt = extract_number(sample["answer"])

    if pred == gt:
        correct += 1

accuracy = correct / total

print("Accuracy:", accuracy)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Accuracy: 0.18


In [ ]:
for i in range(5):

    sample = test_data[i]

    out = teacher_reason(sample["question"])

    print("Q:", sample["question"])
    print("Model Output:", out)
    print("GT:", sample["answer"])
    print("Pred:", extract_number(out))
    print("------")

In [ ]:
agent = AgenticStudent(student_model, tokenizer)

correct = 0

for i in range(50):

    sample = test_data[i]

    pred = agent.run(sample["question"])

    if pred == extract_number(sample["answer"]):
        correct += 1

student_accuracy = correct / 50

print("Phase 2 Student:", student_accuracy)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import torch.nn as nn
import torch

class StudentModel(nn.Module):

    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=4
        )

        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x)

    def generate(self, tokenizer, prompt, max_len=50):

        self.eval()

        tokens = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

        generated = tokens

        for _ in range(max_len):

            out = self.forward(generated)

            probs = torch.softmax(out[:, -1, :], dim=-1)

            next_token = torch.multinomial(probs, num_samples=1)

            generated = torch.cat([generated, next_token], dim=1)

        return tokenizer.decode(generated[0], skip_special_tokens=True)

In [ ]:
vocab_size = tokenizer.vocab_size

student_model = StudentModel(vocab_size).to(device)

optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
distilled = []

for i in tqdm(range(500)):   # ✅ FIX HERE

    sample = train_data[i]

    reasoning = teacher_reason(sample["question"])

    text = f"""
Question: {sample["question"]}
Reasoning: {reasoning}
Answer: {sample["answer"]}
"""

    distilled.append(text)